# Hobby Corner RMS → Lightspeed Product Migration\n**Analysis / dry-run only. This notebook does not write to Lightspeed.**\n\nRMS on-hand quantity is used only as a migration-selection signal. Opening inventory is always 0 here; the physical inventory at cutover establishes actual Lightspeed stock.

In [ ]:
from google.colab import auth\nauth.authenticate_user()\n!pip -q install gspread pandas\n!git clone -q -b product-migration-colab https://github.com/Tawgod/Hobby-Corner-Discord-Bot.git /content/hobby-corner || (cd /content/hobby-corner && git fetch -q && git checkout product-migration-colab && git pull -q)\nimport sys\nsys.path.insert(0, '/content/hobby-corner/migration/product_migration')\n

In [ ]:
import google.auth\nimport gspread\nimport pandas as pd\nfrom analysis import find_category_mapping_sheet, find_supplier_mapping_sheet, build_category_map, build_supplier_map, analyze_candidates, split_outputs, summary\ncreds, _ = google.auth.default()\ngc = gspread.authorize(creds)\n

## Source spreadsheets\nThe RMS candidate sheet is the export from RMS. The LS importer is authoritative for supplier/category destination mappings.

In [ ]:
RMS_CANDIDATE_SHEET = 'https://docs.google.com/spreadsheets/d/1_M4b0iD8EG3pn-cuR_uGZ3KoXL0bA1A_OpusCJboLSQ/edit'\nLS_IMPORTER_SHEET = 'https://docs.google.com/spreadsheets/d/1-F1LpskwwkfiTJzQjpVmZHjOo7iOnCzcFw4CmOiN5kI/edit'\n\ndef workbook_frames(url):\n    book = gc.open_by_url(url)\n    frames = {}\n    for ws in book.worksheets():\n        values = ws.get_all_values()\n        if not values:\n            continue\n        # Find the first nonblank row and use it as headers.\n        header_idx = next((i for i,r in enumerate(values) if any(str(x).strip() for x in r)), 0)\n        headers = values[header_idx]\n        data = values[header_idx+1:]\n        frames[ws.title] = pd.DataFrame(data, columns=headers)\n    return frames\n\nrms_tabs = workbook_frames(RMS_CANDIDATE_SHEET)\nls_tabs = workbook_frames(LS_IMPORTER_SHEET)\nprint('RMS tabs:', list(rms_tabs))\nprint('LS importer tabs:', list(ls_tabs))\n

In [ ]:
# Candidate sheet: automatically choose the first tab containing the core RMS export headers.\nrequired = {'RMS_ItemID','RMS_SKU','RMS_Description','RMS_Department','RMS_Category'}\ncandidate_name, candidates = next((n,d) for n,d in rms_tabs.items() if required.issubset(set(d.columns)))\n\ncategory_tab, category_df = find_category_mapping_sheet(ls_tabs)\nsupplier_tab, supplier_df, supplier_source_col, supplier_target_col = find_supplier_mapping_sheet(ls_tabs)\n\nprint('Candidate tab:', candidate_name)\nprint('Category mapping tab:', category_tab)\nprint('Supplier mapping tab:', supplier_tab, f'({supplier_source_col} -> {supplier_target_col})')\n

In [ ]:
category_map = build_category_map(category_df)\nsupplier_map = build_supplier_map(supplier_df, supplier_source_col, supplier_target_col)\nresult = analyze_candidates(candidates, category_map, supplier_map)\ndisplay(summary(result))\nprint('Unmapped categories:', int((~result['Category_Mapped']).sum()))\nprint('Unmapped suppliers:', int((~result['Supplier_Mapped']).sum()))\nprint('Opening inventory imported:', result['Opening_Inventory_To_Import'].sum())\n

In [ ]:
groups = split_outputs(result)\nfor name, frame in groups.items():\n    print(name, len(frame))\n\n# Focus review queue\nreview_cols = [c for c in ['RMS_ItemID','RMS_SKU','RMS_Description','RMS_Department','RMS_Category','RMS_Supplier','RMS_Quantity','LastSold','LastReceived','LS_Category_Path','LS_Supplier','review_reason'] if c in result.columns]\ndisplay(groups['REVIEW'][review_cols].head(100))\n

## Test-batch selection\nThis only selects candidates. It does **not** create products in Lightspeed.

In [ ]:
# Start with up to 10 fully mapped READY rows. We will intentionally add edge cases before the first real test.\ntest_batch = result[(result['migration_status']=='READY') & result['Category_Mapped'] & result['Supplier_Mapped']].head(10).copy()\ndisplay(test_batch[[c for c in ['RMS_ItemID','RMS_SKU','RMS_Description','LS_Category_Path','LS_Supplier','proposed_reorder_point','proposed_restock_level'] if c in test_batch.columns]])\n